<a href="https://colab.research.google.com/github/M-Baxx/Cardiovascular_LRP1_Expression/blob/main/Image_Quantificaiton_iVSMC_WALL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""Reproducible brightfield attached-cell density analysis.

The count is defined by detected cell-centre events. Optional region traces are
an audit visualization only and do not alter the count. Input image identity,
magnification, well assignment and inclusion decisions are supplied in a CSV
manifest rather than inferred from filenames.

Required manifest columns:
    image_path,batch,sample,well,magnification,include

Optional columns are retained in the output (for example actual_label, notes,
condition, ECM and cell_line). Relative image paths are resolved against
--image-root, or against the manifest directory when --image-root is omitted.
"""

from __future__ import annotations

import argparse
import csv
from collections import defaultdict
from pathlib import Path
from typing import Iterable, List, Mapping, MutableMapping, Optional, Sequence, Tuple

import cv2
import numpy as np
from scipy.ndimage import binary_erosion, distance_transform_edt, maximum_filter


DEFAULT_PPU = {"4x": 0.3087, "10x": 0.782}


def parse_bool(value: object) -> bool:
    return str(value).strip().lower() not in {"", "0", "false", "no", "n", "exclude"}


def normalize_magnification(value: object) -> str:
    text = str(value).strip().lower().replace("×", "x")
    if text in {"4", "4x"}:
        return "4x"
    if text in {"10", "10x"}:
        return "10x"
    raise ValueError(f"Unsupported magnification: {value!r}; expected 4x or 10x")


def read_manifest(path: Path, image_root: Optional[Path]) -> Tuple[List[dict], List[dict]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        required = {"image_path", "batch", "sample", "well", "magnification", "include"}
        missing = required.difference(reader.fieldnames or [])
        if missing:
            raise ValueError(f"Manifest is missing columns: {', '.join(sorted(missing))}")
        records = list(reader)

    root = image_root if image_root is not None else path.parent
    included: List[dict] = []
    excluded: List[dict] = []
    for row_number, raw in enumerate(records, start=2):
        row = {str(k): v for k, v in raw.items()}
        row["manifest_row"] = row_number
        if not parse_bool(row.get("include")):
            row["analysis_status"] = "excluded_by_manifest"
            excluded.append(row)
            continue
        try:
            row["magnification"] = normalize_magnification(row["magnification"])
        except ValueError as exc:
            row["analysis_status"] = f"excluded_invalid_magnification: {exc}"
            excluded.append(row)
            continue
        image_path = Path(str(row["image_path"]).strip())
        if not image_path.is_absolute():
            image_path = root / image_path
        row["resolved_image_path"] = str(image_path.resolve())
        if not image_path.is_file():
            row["analysis_status"] = "excluded_missing_file"
            excluded.append(row)
            continue
        row["analysis_status"] = "included"
        included.append(row)
    return included, excluded


def load_grayscale(path: str) -> np.ndarray:
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise RuntimeError(f"Could not read image: {path}")
    if gray.ndim != 2:
        raise RuntimeError(f"Expected a two-dimensional grayscale image: {path}")
    return gray


def detect_centres(
    gray: np.ndarray,
    pixels_per_um: float,
    response_percentile: float,
    small_sigma_um: float,
    background_sigma_um: float,
    texture_sigma_um: float,
    texture_sd_threshold: float,
    minimum_spacing_um: float,
) -> dict:
    """Detect attached cell-centre events using the study's fixed-scale method."""
    image = gray.astype(np.float32) / 255.0
    response = cv2.GaussianBlur(
        image, (0, 0), background_sigma_um * pixels_per_um
    ) - cv2.GaussianBlur(image, (0, 0), small_sigma_um * pixels_per_um)

    gray_float = gray.astype(np.float32)
    texture_sigma_px = texture_sigma_um * pixels_per_um
    local_mean = cv2.GaussianBlur(gray_float, (0, 0), texture_sigma_px)
    local_mean_sq = cv2.GaussianBlur(gray_float**2, (0, 0), texture_sigma_px)
    local_sd = np.sqrt(np.maximum(0.0, local_mean_sq - local_mean**2))
    attached_region = local_sd > texture_sd_threshold

    nms_size = max(3, int(round(minimum_spacing_um * pixels_per_um)))
    if nms_size % 2 == 0:
        nms_size += 1
    border = max(12, nms_size)
    if gray.shape[0] <= 2 * border or gray.shape[1] <= 2 * border:
        raise ValueError("Image is too small for the configured border and spacing")

    threshold = float(
        np.percentile(response[border:-border, border:-border], response_percentile)
    )
    maxima = maximum_filter(response, size=nms_size, mode="nearest")
    detected = (response == maxima) & (response > threshold) & attached_region
    detected[:border, :] = False
    detected[-border:, :] = False
    detected[:, :border] = False
    detected[:, -border:] = False
    y, x = np.nonzero(detected)

    area_cm2 = gray.shape[0] * gray.shape[1] / (pixels_per_um**2) / 1.0e8
    count = int(len(x))
    density = count / area_cm2
    return {
        "x": x,
        "y": y,
        "response": response,
        "attached_region": attached_region,
        "count": count,
        "field_area_cm2": float(area_cm2),
        "density_cells_cm2": float(density),
        "focus_score_laplacian_variance": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        "contrast_sd": float(np.std(gray)),
        "attached_area_fraction": float(attached_region.mean()),
        "response_threshold": threshold,
        "nms_window_px": nms_size,
    }


def centre_overlay(gray: np.ndarray, result: Mapping[str, object], mag: str) -> np.ndarray:
    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    overlay = rgb.copy()
    radius = 2 if mag == "4x" else 4
    for x, y in zip(result["x"], result["y"]):
        cv2.circle(overlay, (int(x), int(y)), radius, (0, 230, 255), 1, cv2.LINE_AA)
    return cv2.addWeighted(rgb, 0.68, overlay, 0.32, 0.0)


def traced_region_overlay(
    gray: np.ndarray,
    result: Mapping[str, object],
    pixels_per_um: float,
    mag: str,
) -> np.ndarray:
    """Draw audit regions assigned to centres; these are not membrane boundaries."""
    x = np.asarray(result["x"])
    y = np.asarray(result["y"])
    response = np.asarray(result["response"])
    attached = np.asarray(result["attached_region"], dtype=bool)
    if len(x) == 0 or not attached.any():
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    seed_mask = np.zeros(gray.shape, dtype=bool)
    seed_mask[y, x] = True
    seed_labels = np.zeros(gray.shape, dtype=np.int32)
    seed_labels[y, x] = np.arange(1, len(x) + 1, dtype=np.int32)
    distance, nearest = distance_transform_edt(~seed_mask, return_indices=True)
    labels = seed_labels[nearest[0], nearest[1]]
    response_floor = float(np.percentile(response[attached], 30.0))
    support = attached & (distance <= 18.0 * pixels_per_um) & (response > response_floor)
    labels = np.where(support, labels, 0)
    boundary = support & ~binary_erosion(support, structure=np.ones((3, 3)), border_value=0)
    boundary[:, 1:] |= support[:, 1:] & (labels[:, 1:] != labels[:, :-1])
    boundary[1:, :] |= support[1:, :] & (labels[1:, :] != labels[:-1, :])
    boundary = cv2.dilate(boundary.astype(np.uint8), np.ones((2, 2), np.uint8)) > 0

    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    overlay = rgb.copy()
    overlay[boundary] = (255, 229, 0)
    radius = 1 if mag == "4x" else 2
    for xx, yy in zip(x, y):
        cv2.circle(overlay, (int(xx), int(yy)), radius, (0, 190, 255), -1, cv2.LINE_AA)
    return cv2.addWeighted(rgb, 0.38, overlay, 0.62, 0.0)


def safe_stem(row: Mapping[str, object], index: int) -> str:
    raw = f"{row.get('batch','batch')}_s{row.get('sample','sample')}_w{row.get('well','well')}_{row['magnification']}_{index:03d}"
    return "".join(c if c.isalnum() or c in "-_" else "_" for c in raw)


def write_csv(path: Path, rows: Sequence[Mapping[str, object]]) -> None:
    fields: List[str] = []
    for row in rows:
        for key in row:
            if key not in fields:
                fields.append(key)
    with path.open("w", newline="", encoding="utf-8") as handle:
        if not fields:
            handle.write("")
            return
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def mean(values: Iterable[float]) -> float:
    values = list(values)
    return float(sum(values) / len(values)) if values else float("nan")


def aggregate_wells(image_rows: Sequence[Mapping[str, object]], well_area_cm2: float) -> List[dict]:
    grouped: MutableMapping[Tuple[str, str, str, str], List[Mapping[str, object]]] = defaultdict(list)
    for row in image_rows:
        key = (str(row["batch"]), str(row["sample"]), str(row["well"]), str(row["magnification"]))
        grouped[key].append(row)

    by_well: MutableMapping[Tuple[str, str, str], dict] = {}
    for (batch, sample, well, mag), fields in grouped.items():
        key = (batch, sample, well)
        output = by_well.setdefault(key, {"batch": batch, "sample": sample, "well": well})
        prefix = mag.replace("x", "x_")
        output[f"{prefix}image_fields"] = len(fields)
        output[f"{prefix}mean_count"] = mean(float(r["count"]) for r in fields)
        output[f"{prefix}mean_field_area_cm2"] = mean(float(r["field_area_cm2"]) for r in fields)
        output[f"{prefix}mean_density_cells_cm2"] = mean(float(r["density_cells_cm2"]) for r in fields)
        output[f"{prefix}projected_yield_cells"] = output[f"{prefix}mean_density_cells_cm2"] * well_area_cm2
        output[f"{prefix}source_images"] = " | ".join(str(r["image_path"]) for r in fields)

    rows = [by_well[key] for key in sorted(by_well)]
    for row in rows:
        four = row.get("4x_mean_density_cells_cm2")
        ten = row.get("10x_mean_density_cells_cm2")
        if four is not None and ten is not None and float(four) != 0:
            row["ratio_10x_to_4x"] = float(ten) / float(four)
            row["symmetric_difference_percent"] = (
                (float(ten) - float(four)) / ((float(ten) + float(four)) / 2.0) * 100.0
            )
    return rows


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--manifest", type=Path, required=True, help="CSV image manifest")
    parser.add_argument("--output-dir", type=Path, required=True, help="Directory for CSV outputs and overlays")
    parser.add_argument("--image-root", type=Path, default=None, help="Root for relative manifest image paths")
    parser.add_argument("--ppu-4x", type=float, default=DEFAULT_PPU["4x"])
    parser.add_argument("--ppu-10x", type=float, default=DEFAULT_PPU["10x"])
    parser.add_argument("--well-area-cm2", type=float, default=9.6)
    parser.add_argument("--response-percentile", type=float, default=80.0)
    parser.add_argument("--sensitivity-low-percentile", type=float, default=75.0)
    parser.add_argument("--sensitivity-high-percentile", type=float, default=85.0)
    parser.add_argument("--small-sigma-um", type=float, default=3.2)
    parser.add_argument("--background-sigma-um", type=float, default=16.0)
    parser.add_argument("--texture-sigma-um", type=float, default=40.0)
    parser.add_argument("--texture-sd-threshold", type=float, default=6.0)
    parser.add_argument("--minimum-spacing-um", type=float, default=22.0)
    parser.add_argument("--no-overlays", action="store_true", help="Do not save centre overlays")
    parser.add_argument("--region-traces", action="store_true", help="Also save optional assigned-region trace overlays")
    return parser


def main() -> None:
    args = build_parser().parse_args()
    args.output_dir.mkdir(parents=True, exist_ok=True)
    overlay_dir = args.output_dir / "centre_overlays"
    trace_dir = args.output_dir / "region_trace_overlays"
    if not args.no_overlays:
        overlay_dir.mkdir(parents=True, exist_ok=True)
    if args.region_traces:
        trace_dir.mkdir(parents=True, exist_ok=True)

    records, excluded = read_manifest(args.manifest, args.image_root)
    ppu = {"4x": args.ppu_4x, "10x": args.ppu_10x}
    image_rows: List[dict] = []
    for index, record in enumerate(records, start=1):
        mag = str(record["magnification"])
        gray = load_grayscale(str(record["resolved_image_path"]))
        settings = dict(
            pixels_per_um=ppu[mag],
            small_sigma_um=args.small_sigma_um,
            background_sigma_um=args.background_sigma_um,
            texture_sigma_um=args.texture_sigma_um,
            texture_sd_threshold=args.texture_sd_threshold,
            minimum_spacing_um=args.minimum_spacing_um,
        )
        result = detect_centres(gray, response_percentile=args.response_percentile, **settings)
        low = detect_centres(gray, response_percentile=args.sensitivity_low_percentile, **settings)["count"]
        high = detect_centres(gray, response_percentile=args.sensitivity_high_percentile, **settings)["count"]
        stem = safe_stem(record, index)

        row = dict(record)
        row.update({k: v for k, v in result.items() if k not in {"x", "y", "response", "attached_region"}})
        row["pixels_per_um"] = ppu[mag]
        row["projected_yield_cells"] = float(result["density_cells_cm2"]) * args.well_area_cm2
        row["sensitivity_count_low"] = min(int(low), int(high))
        row["sensitivity_count_high"] = max(int(low), int(high))
        row["sensitivity_density_low_cells_cm2"] = row["sensitivity_count_low"] / float(result["field_area_cm2"])
        row["sensitivity_density_high_cells_cm2"] = row["sensitivity_count_high"] / float(result["field_area_cm2"])

        if not args.no_overlays:
            centre_path = overlay_dir / f"{stem}.png"
            cv2.imwrite(str(centre_path), centre_overlay(gray, result, mag))
            row["centre_overlay_path"] = str(centre_path)
        if args.region_traces:
            trace_path = trace_dir / f"{stem}.png"
            cv2.imwrite(str(trace_path), traced_region_overlay(gray, result, ppu[mag], mag))
            row["region_trace_overlay_path"] = str(trace_path)
        image_rows.append(row)
        print(f"[{index}/{len(records)}] {Path(record['resolved_image_path']).name}: {result['count']:,} centres")

    well_rows = aggregate_wells(image_rows, args.well_area_cm2)
    write_csv(args.output_dir / "image_results.csv", image_rows)
    write_csv(args.output_dir / "well_results.csv", well_rows)
    write_csv(args.output_dir / "excluded_manifest_rows.csv", excluded)
    print(f"Completed {len(image_rows)} images; {len(excluded)} manifest rows excluded")


if __name__ == "__main__":
    main()
